In [ ]:
"""
Data Collection

Data source : https://data.police.uk/api
Boundaries  : London borough GeoJSON
Output      : data/raw/crime_data_london_raw.csv
              data/raw/crime_monthly_by_borough.csv
              data/raw/crime_categories_by_borough.csv
"""

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime

os.makedirs("data/raw", exist_ok=True)

#  Date range
START_DATE = "2023-01"
END_DATE   = "2026-2"

BASE_URL   = "https://data.police.uk/api/crimes-street/all-crime"

# Downloading borough boundaries from London Datastore
GEOJSON_URL = (
    "https://raw.githubusercontent.com/radoi90/housequest-data/master/"
    "london_boroughs.geojson"
)

def get_borough_polygons() -> dict:
    
    #Returns {borough_name: [(lng, lat), ...]}

    cache_path = "data/raw/london_boroughs.geojson"

    if os.path.exists(cache_path):
        print("Loading cached borough boundaries...")
        with open(cache_path) as f:
            geojson = json.load(f)
    else:
        print("Downloading borough boundaries...")
        r = requests.get(GEOJSON_URL, timeout=30)
        r.raise_for_status()
        geojson = r.json()
        with open(cache_path, "w") as f:
            json.dump(geojson, f)
        print(f"Saved to {cache_path}")

    polygons = {}
    for feature in geojson["features"]:
        name = feature["properties"].get("name") or feature["properties"].get("NAME") or feature["properties"].get("lad11nm", "")
        coords = feature["geometry"]["coordinates"]
        geom_type = feature["geometry"]["type"]

        # Extract outer ring of polygon
        if geom_type == "Polygon":
            ring = coords[0]
        elif geom_type == "MultiPolygon":
            ring = max(coords, key=lambda p: len(p[0]))[0]
        else:
            continue

        polygons[name] = ring  # list of [lng, lat]

    print(f"Loaded boundaries for {len(polygons)} boroughs\n")
    return polygons


# Simplifying polygon for API (max ~1000 points, API limit) 
def simplify_polygon(coords: list, max_points: int = 50) -> list:
    #Reduces polygon to max_points using uniform sampling.
    
    if len(coords) <= max_points:
        return coords
    step = len(coords) // max_points
    simplified = coords[::step]
    if simplified[0] != simplified[-1]:
        simplified.append(simplified[0])
    return simplified


# Formating polygon for API 
def format_poly(coords: list) -> str:
    return ":".join(f"{lat},{lng}" for lng, lat in coords)


# Generating month list 
def date_range(start: str, end: str) -> list:
    start_dt = datetime.strptime(start, "%Y-%m")
    end_dt   = datetime.strptime(end,   "%Y-%m")
    months, current = [], start_dt
    while current <= end_dt:
        months.append(current.strftime("%Y-%m"))
        current = current.replace(month=current.month % 12 + 1,
                                  year=current.year + (1 if current.month == 12 else 0))
    return months


# Main download loop
def download_crime_data(polygons: dict) -> pd.DataFrame:
    months = date_range(START_DATE, END_DATE)
    all_records = []
    total = len(polygons) * len(months)
    done  = 0

    print(f"Downloading: {len(polygons)} boroughs x {len(months)} months = {total} requests")
    print(f"Estimated time: ~{round(total * 2 / 60)} minutes\n")

    for borough, coords in polygons.items():
        simplified = simplify_polygon(coords, max_points=50)
        poly_str   = format_poly(simplified)

        for month in months:
            done += 1
            params = {"poly": poly_str, "date": month}

            try:
                response = requests.get(BASE_URL, params=params, timeout=30)

                if response.status_code == 200:
                    crimes = response.json()
                    for crime in crimes:
                        all_records.append({
                            "borough":        borough,
                            "month":          month,
                            "category":       crime.get("category", ""),
                            "location_type":  crime.get("location_type", ""),
                            "latitude":       crime.get("location", {}).get("latitude", ""),
                            "longitude":      crime.get("location", {}).get("longitude", ""),
                            "street":         crime.get("location", {}).get("street", {}).get("name", ""),
                            "outcome_status": (crime.get("outcome_status") or {}).get("category", "No outcome"),
                            "persistent_id":  crime.get("persistent_id", ""),
                            "id":             crime.get("id", ""),
                        })
                    print(f"[{done:>4}/{total}] {borough:<35} {month}  →  {len(crimes):>5} crimes")

                elif response.status_code == 503:
                    print(f"[{done:>4}/{total}] {borough:<35} {month}  →  503 (too many crimes, skipping)")

                elif response.status_code == 429:
                    print("  Rate limited — waiting 15s...")
                    time.sleep(15)
                    response = requests.get(BASE_URL, params=params, timeout=30)
                    if response.status_code == 200:
                        crimes = response.json()
                        for crime in crimes:
                            all_records.append({
                                "borough": borough, "month": month,
                                "category": crime.get("category", ""),
                                "id": crime.get("id", ""),
                            })

                else:
                    print(f"  Warning: {borough} {month} → HTTP {response.status_code}")

            except requests.exceptions.RequestException as e:
                print(f"  Error: {borough} {month} — {e}")

            time.sleep(1.5) 

        # Save checkpoint every borough in case of interruption
        checkpoint = pd.DataFrame(all_records)
        checkpoint.to_csv("data/raw/crime_data_checkpoint.csv", index=False)

    return pd.DataFrame(all_records)


#  Aggregate 
def aggregate(df: pd.DataFrame):
    monthly = (
        df.groupby(["borough", "month"])
          .agg(crime_count=("id", "count"))
          .reset_index()
    )
    category_pivot = (
        df.groupby(["borough", "month", "category"])
          .size()
          .unstack(fill_value=0)
          .reset_index()
    )
    return monthly, category_pivot


# Run 
if __name__ == "__main__":
    print("=" * 60)
    print("  Phase 1 — UK Police API (polygon-based, FIXED)")
    print("=" * 60)
    print(f"  Date range : {START_DATE}  →  {END_DATE}")
    print("=" * 60 + "\n")

    # Get borough polygons
    polygons = get_borough_polygons()

    # Download
    df_raw = download_crime_data(polygons)

    # Save raw
    raw_path = "data/raw/crime_data_london_raw.csv"
    df_raw.to_csv(raw_path, index=False)
    print(f"\nRaw data saved     →  {raw_path}")
    print(f"Total records      :  {len(df_raw):,}")

    # Aggregate
    df_monthly, df_categories = aggregate(df_raw)
    df_monthly.to_csv("data/raw/crime_monthly_by_borough.csv", index=False)
    df_categories.to_csv("data/raw/crime_categories_by_borough.csv", index=False)
    print(f"Monthly counts     →  data/raw/crime_monthly_by_borough.csv")
    print(f"Category pivot     →  data/raw/crime_categories_by_borough.csv")

    # Summary
    print("\n Summary")
    print(f"Date range   : {df_monthly['month'].min()}  →  {df_monthly['month'].max()}")
    print(f"Boroughs     : {df_monthly['borough'].nunique()}")
    print(f"Total crimes : {df_monthly['crime_count'].sum():,}")
    print(f"\nTop 5 boroughs by total crime:")
    top5 = df_monthly.groupby("borough")["crime_count"].sum().sort_values(ascending=False).head()
    for b, c in top5.items():
        print(f"  {b:<35} {c:,}")
    print("\nDone.")

  Phase 1 — UK Police API (polygon-based, FIXED)
  Date range : 2021-01  →  2026-2

Saved to data/raw/london_boroughs.geojson
Loaded boundaries for 33 boroughs

Downloading: 33 boroughs x 62 months = 2046 requests
Estimated time: ~68 minutes

[  27/2046] Barking and Dagenham                2023-03  →   2036 crimes
[  28/2046] Barking and Dagenham                2023-04  →   2001 crimes
[  29/2046] Barking and Dagenham                2023-05  →   2200 crimes
[  30/2046] Barking and Dagenham                2023-06  →   2344 crimes
[  31/2046] Barking and Dagenham                2023-07  →   2282 crimes
[  32/2046] Barking and Dagenham                2023-08  →   2144 crimes
[  33/2046] Barking and Dagenham                2023-09  →   2235 crimes
[  34/2046] Barking and Dagenham                2023-10  →   1798 crimes
[  35/2046] Barking and Dagenham                2023-11  →   1882 crimes
[  36/2046] Barking and Dagenham                2023-12  →   2028 crimes
[  37/2046] Barking and Dag